In [ ]:
import numpy as np
import pandas as pd
import math
import random
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn import ensemble  
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor 
from sklearn.neural_network import MLPRegressor 
from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
merged_data = pd.read_csv(r"D:\FPL\fpl-optimization\point-predictor\code\xP Model\data\all_players_timeseries_dataset.csv", index_col=False)

In [ ]:
filtered_data = merged_data[(merged_data['match_count']<5) | (merged_data['GW']<5)]

In [ ]:
def plot_nas(df: pd.DataFrame):
    if df.isnull().sum().sum() != 0:
        na_df = (df.isnull().sum() / len(df)) * 100      
        na_df = na_df.drop(na_df[na_df == 0].index).sort_values(ascending=False)
        missing_data = pd.DataFrame({'Missing Ratio %' :na_df})
        missing_data.plot(kind = "barh")
        plt.show()
    else:
        print('No NAs found')
plot_nas(filtered_data)
plot_width, plot_height = (10,10)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)

In [ ]:
# Drop columns that contain any null values
cleaned_data = filtered_data.dropna(axis=1)

# Display the resulting DataFrame
cleaned_data

In [ ]:
# Define a mapping of position names to numerical codes
position_mapping = {'GK': 1, 'GKP':1, 'DEF': 2, 'MID': 3, 'FWD': 4}

# Encode the "position" column using the mapping
cleaned_data['position_encoded'] = cleaned_data['position'].replace(position_mapping)


In [ ]:
cleaned_data['was_home'] = cleaned_data['was_home'].astype('bool')
# Use mapping of team names to numerical codes based on reverse order of PL League table total points
team_key_df = pd.read_csv(r"data/team_key.csv", index_col=False)
team_name_to_code = team_key_df.set_index('team')['team_code'].to_dict()

cleaned_data['team_x'] = cleaned_data['team_x'].replace(team_name_to_code)
cleaned_data['opponent'] = cleaned_data['opp_team_name'].replace(team_name_to_code)

In [ ]:
train = cleaned_data[cleaned_data["season_x"]!="2024-25"]
test = cleaned_data[cleaned_data["season_x"]=="2024-25"]

In [ ]:
test.tail()

In [ ]:
train_num = train.drop(['season_x','name','GW','position','opp_team_name','total_points'], axis=1)
test_num = test.drop(['season_x','name','GW','position','opp_team_name','total_points'], axis=1)

In [ ]:
train_num.tail()

In [ ]:
# Baseline model
predict_train = np.mean(train.total_points)
# Get predictions on the test set
predict_test = np.ones(test.total_points.shape) * predict_train
residuals = predict_train - train.total_points
residuals2 = predict_test - test.total_points
mse_train = np.sqrt(sum(residuals**2)/len(residuals))
mse_test = np.sqrt(sum(residuals2**2)/len(residuals2))
abs_train = sum(abs(residuals))/len(residuals)
abs_test = sum(abs(residuals2))/len(residuals2)
print('sqrt(MSE) on train set: ', mse_train)
print('sqrt(MSE) on test set: ', mse_test)
print('Mean Absolute value of residuals on train set: ', abs_train)
print('Mean Absolute value of residuals on test set: ', abs_test)

In [ ]:

# Sample data for demonstration
# Each row represents a player with attributes: points, value, position, team rating, matches played
# player_data = np.array([
#     [150, 10.0, 1, 4.5, 20],
#     [120, 9.5, 2, 4.3, 21],
#     [180, 11.5, 3, 4.8, 22],
#     [90, 5.0, 4, 3.9, 18],
#     [200, 12.0, 1, 4.9, 23],
#     [110, 7.5, 2, 4.0, 19]
# ])

# Standardize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(train_num)

# Specify the number of clusters
num_clusters = 5

# Perform K-means clustering
kmeans = KMeans(n_clusters=num_clusters)
kmeans.fit(train_num)

# Assign players to clusters
player_clusters = kmeans.predict(train_num)




In [ ]:
Sum_of_squared_distances = []
K = range(1,50)
for num_clusters in K :
 kmeans = KMeans(n_clusters=num_clusters)
 kmeans.fit(scaled_data)
 Sum_of_squared_distances.append(kmeans.inertia_)
plt.plot(K,Sum_of_squared_distances,'bx-')
plt.xlabel('Values of K') 
plt.ylabel('Sum of squared distances/Inertia') 
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:
range_n_clusters = range(3,50)
silhouette_avg = []
for num_clusters in range_n_clusters:
 
     # initialise kmeans
    kmeans = KMeans(n_clusters=num_clusters)
    kmeans.fit(scaled_data)
    cluster_labels = kmeans.labels_

     # silhouette score
    silhouette_avg.append(silhouette_score(scaled_data, cluster_labels))
plt.plot(range_n_clusters,silhouette_avg,'bx-')
plt.xlabel('Values of K') 
plt.ylabel('Silhouette score') 
plt.title('Silhouette analysis For Optimal k')
plt.show()

In [ ]:
optimal_clusters = range_n_clusters[silhouette_avg.index(max(silhouette_avg))]
print(f'The optimal number of clusters is: {optimal_clusters}')

In [ ]:
silhouette_avg[45]

In [ ]:
# Standardize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(train_num)

# Specify the number of clusters
num_clusters = optimal_clusters

# Perform K-means clustering
kmeans = KMeans(n_clusters=num_clusters)
kmeans.fit(train_num)

# Assign players to clusters
player_clusters = kmeans.predict(train_num)




In [ ]:
test_num[test['name']=='Erling Haaland'].head()

In [ ]:
# New player attributes (points, value, position, team rating, matches played)
new_player_attributes = test_num[test['name']=='Erling Haaland'].head(1)
# Standardize new player attributes
scaled_new_player = scaler.transform(new_player_attributes)

# Predict cluster for new player
new_player_cluster = kmeans.predict(scaled_new_player)
new_player_cluster = kmeans.predict(new_player_attributes)

# Find cluster members
cluster_members = np.where(player_clusters == new_player_cluster)[0]

# Estimate points for the new player based on cluster members' average
estimated_points = np.mean(train['total_points'].iloc[cluster_members])

# # Plotting
# plt.figure(figsize=(10, 6))
# colors = ['r', 'g', 'b','yellow','grey','orange']  # Color for each position
# for i, label in enumerate(np.unique(train[:, 7])):
#     indices = np.where(player_data[:, 7] == label)
#     plt.scatter(train[indices, 3], train[indices, 4], color=colors[i], label=f'Position {int(label)}')
# plt.scatter(new_player_attributes[3, 4], estimated_points, color='purple', marker='*', s=100, label='New Player')
# plt.xlabel('Matches Played')
# plt.ylabel('Points Scored')
# plt.title('Player Clusters based on Matches Played and Points Scored')
# plt.legend()
# plt.grid()
# plt.show()

print(f"Estimated points for the new player: {estimated_points}")

In [ ]:
train.iloc[cluster_members]

In [ ]:
# #Getting unique labels
# centroids = kmeans.cluster_centers_
# u_labels = np.unique(player_clusters)
 
# #plotting the results:
 
# for i in u_labels:
#     plt.scatter(train[player_clusters == i]['total_points'] , train[player_clusters == i]['value'] , label = i)
# plt.scatter(estimated_points , centroids[:,4] , s = 80, color = 'black')
# plt.legend()
# plt.show()

# Assuming train, player_clusters, kmeans, and estimated_points are defined

centroids = kmeans.cluster_centers_
u_labels = np.unique(player_clusters)

# Plotting the results:
for i in u_labels:
    plt.scatter(train[player_clusters == i]['total_points'], train[player_clusters == i]['value'], label=f'Cluster {i}')

# Ensure estimated_points is defined and matches the length of centroids[:,4]
# For demonstration, let's assume estimated_points has the same length as centroids[:,4]
# If not, you need to adjust this accordingly

if len(estimated_points) == len(centroids[:, 4]):
    plt.scatter(estimated_points, centroids[:, 4], s=80, color='black')
else:
    print("Error: The length of estimated_points does not match the number of centroids.")

plt.legend()
plt.show()

In [ ]:
train[player_clusters].value_counts()

In [ ]:

new_player_attributes = np.array(test_num)
# Standardize new player attributes
scaled_new_player = scaler.transform(new_player_attributes)

# Predict cluster for new player
new_player_cluster = kmeans.predict(scaled_new_player)
new_player_cluster = kmeans.predict(new_player_attributes)



In [ ]:
test['cluster'] = new_player_cluster

In [ ]:
test.head()

In [ ]:
estimated_points = np.zeros(len(np.unique(player_clusters)))
for i in np.unique(player_clusters):
    # Find cluster members
    cluster_members = np.where(player_clusters == i)[0]

    # Estimate points for the new player based on cluster members' average
    estimated_points[i] = np.mean(train['total_points'].iloc[cluster_members])

    print(f"Estimated points for the new player: {estimated_points[i]}")

In [ ]:
test['xP'] = estimated_points[test['cluster']]

In [ ]:
test.head()

In [ ]:
residuals = test['xP'] - test['total_points']

mse_predict = np.sqrt(sum(residuals**2)/len(residuals))

abs_predict = sum(abs(residuals))/len(residuals)


print('sqrt(MSE) on prediction set: ', mse_predict)

print('Mean Absolute value of residuals on prediction set: ', abs_predict)

In [ ]:
test.to_csv('datasets\\new_player_predictions.csv',index=False)

In [ ]:
train.iloc[np.where(player_clusters == 14)]

In [ ]:
# Calculate average total_points for each cluster
cluster_avg_total_points = []
for i in range(optimal_clusters):
    cluster_labels = kmeans.labels_
    cluster_avg = np.mean(train[cluster_labels == i]['total_points'])
    cluster_avg_total_points.append(cluster_avg)

# Plot bar chart
plt.bar(range(optimal_clusters), cluster_avg_total_points)
plt.xlabel('Cluster')
plt.ylabel('Average Total Points')
plt.title('Average Total Points by Cluster')
plt.show()

In [ ]:
train.head(5)

In [ ]:
# Create a dictionary to store player information for each cluster
cluster_players = {}
for i in range(optimal_clusters):
    cluster_players[i] = []

# Populate the dictionary with player information
for index, row in train.iterrows():
    cluster_label = kmeans.labels_[index]
    player_info = {
        'name': row['name'],
        'position': row['position'],
        'value': row['value']
    }
    cluster_players[cluster_label].append(player_info)

# Print player information for each cluster
for cluster, players in cluster_players.items():
    print(f"Cluster {cluster}:")
    for player in players:
        print(f"  - {player['name']} ({player['position']}) - Value: {player['value']}")
    print()